In [ ]:
import json
import random
from collections import defaultdict

import pandas as pd

with open("/content/train_mini.json") as file:
    train_data = json.load(file)

with open("/content/val.json") as file:
    test_data = json.load(file)


def group_by_class(data):
    paths = {}
    for image in data["images"]:
        paths[image["id"]] = image["file_name"]

    groups = defaultdict(list)
    for annotation in data["annotations"]:
        groups[annotation["category_id"]].append(paths[annotation["image_id"]])

    return groups


train_groups = group_by_class(train_data)
test_groups = group_by_class(test_data)

eligible = []
for category_id in train_groups:
    if len(train_groups[category_id]) >= 50 and len(test_groups[category_id]) >= 10:
        eligible.append(category_id)

rng = random.Random(42)
selected = sorted(rng.sample(eligible, 500))

rows = []
for label, category_id in enumerate(selected):
    images = train_groups[category_id]
    rng.shuffle(images)

    for path in images[:40]:
        rows.append(("train", label, category_id, path))

    for path in images[40:50]:
        rows.append(("validation", label, category_id, path))

    for path in test_groups[category_id][:10]:
        rows.append(("test", label, category_id, path))

manifest = pd.DataFrame(
    rows,
    columns=["split", "label", "category_id", "file_name"]
)

names = {}
for category in train_data["categories"]:
    names[category["id"]] = category["name"]

class_rows = []
for label, category_id in enumerate(selected):
    class_rows.append((label, category_id, names[category_id]))

classes = pd.DataFrame(
    class_rows,
    columns=["label", "category_id", "name"]
)

manifest.to_csv("dataset_manifest.csv", index=False)
classes.to_csv("selected_classes.csv", index=False)

print(manifest["split"].value_counts())
print("Classes:", len(classes))

In [ ]:
!wget -c https://ml-inat-competition-datasets.s3.amazonaws.com/2021/train_mini.tar.gz
!wget -c https://ml-inat-competition-datasets.s3.amazonaws.com/2021/val.tar.gz

In [ ]:
import tarfile
from pathlib import Path

import pandas as pd

manifest = pd.read_csv("dataset_manifest.csv")
output = Path("selected_images")
output.mkdir(exist_ok=True)


def extract_selected(archive_name, selected_paths):
    selected_paths = set(selected_paths)

    with tarfile.open(archive_name, "r:gz") as archive:
        for member in archive:
            path = member.name.removeprefix("./")

            if member.isfile() and path in selected_paths:
                archive.extract(member, output, filter="data")


extract_selected(
    "train_mini.tar.gz",
    manifest[manifest["split"] != "test"]["file_name"]
)

extract_selected(
    "val.tar.gz",
    manifest[manifest["split"] == "test"]["file_name"]
)

number_of_images = sum(
    path.is_file()
    for path in output.rglob("*")
)

print(number_of_images)

In [ ]:
import shutil

shutil.make_archive(
    "selected_images",
    "gztar",
    root_dir="selected_images"
)